# Forest Memory — LiteRT Sensing + Gemma 4 Reasoning

**Hackathon:** Kaggle Gemma 4 Good — Special Technology Track: LiteRT

---

## Architecture

```
🎤 WAV audio ──→  LiteRT / YAMNet  ──→  bird:0.82  insect:0.61  human:0.09  ─┐
                  (sense: ear)                                                 │
🛰️  RGB TIF   ──→  pixel analysis   ──→  green_dominance  burned_signal        ├──→ Gemma 4 ──→ Report
                  (sense: eye)                                                 │    (brain)
📊 NDVI + metadata ───────────────────────────────────────────────────────────┘
```

**LiteRT = sensory organs** (ear + eye), **Gemma 4 = reasoning brain**

Both YAMNet and Gemma 4 run via LiteRT — fully offline on Kaggle or edge hardware.

---

### Scientific constraints
- **DO:** proxy-based ecological reasoning, multimodal signal interpretation, uncertainty-aware reporting
- **DO NOT:** claim exact species counts, predict wildfire, diagnose ecosystem collapse


## Step 1 — Install dependencies


In [ ]:
import subprocess
result = subprocess.run(["find", "/kaggle/input", "-type", "f"], 
                       capture_output=True, text=True)
print(result.stdout[:3000] or "EMPTY — no files found")


In [ ]:
# Install required packages
import subprocess, sys

def pip(pkg):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

pip("ai-edge-litert>=1.0")   # LiteRT runtime
pip("mediapipe>=0.10.14")    # LLM Inference API for Gemma 4
pip("soundfile>=0.12")
pip("scipy>=1.11")
pip("rasterio>=1.3")         # GeoTIFF reading (satellite imagery)

print("Setup complete")


In [ ]:
import json, os, time
from pathlib import Path
from IPython.display import display, Markdown
import numpy as np

DATASET_ROOT = Path("/kaggle/input/datasets/zeynepucuncuoglu/forest-memory-data/forest-memory")

CASES_JSON  = DATASET_ROOT / "outputs/forest_memory_cases.json"
AUDIO_ROOT  = DATASET_ROOT / "data/bioscape/audio"
SAT_ROOT    = DATASET_ROOT / "data/bioscape/sattelite"

GEMMA_MODEL = None
for p in Path("/kaggle/input").rglob("*.task"):
    GEMMA_MODEL = p
    break

YAMNET_PATH = Path("/kaggle/working/yamnet.tflite")
OUT_DIR = Path("/kaggle/working/forest_memory_outputs")
OUT_DIR.mkdir(exist_ok=True)

print("cases.json :", CASES_JSON.exists(), CASES_JSON)
print("audio root :", AUDIO_ROOT.exists(), AUDIO_ROOT)
print("sat root   :", SAT_ROOT.exists(),   SAT_ROOT)
print("Gemma      :", GEMMA_MODEL or "Not found — API mode")


## Step 2 — Download YAMNet (LiteRT audio model)


In [ ]:
import urllib.request

YAMNET_URL = (
    "https://storage.googleapis.com/download.tensorflow.org/models/"
    "tflite/task_library/audio_classification/android/"
    "lite-model_yamnet_classification_tflite_1.tflite"
)

if not YAMNET_PATH.exists():
    print("Downloading YAMNet (~3 MB)...")
    urllib.request.urlretrieve(YAMNET_URL, YAMNET_PATH)
    print(f"Downloaded -> {YAMNET_PATH}")
else:
    print(f"YAMNet already present -> {YAMNET_PATH}")

size_mb = YAMNET_PATH.stat().st_size / 1e6
print(f"   Size: {size_mb:.1f} MB")


In [ ]:
# LiteRT Eye: EfficientNet-Lite0 visual texture model
# Strategy: check Kaggle dataset first, then try URLs

DATASET_MODEL_PATH = DATASET_ROOT / "models" / "efficientnet_lite0.tflite"
EFFICIENTNET_WORKING = Path("/kaggle/working/efficientnet_lite0.tflite")

EFFICIENTNET_FALLBACK_URLS = [
    "https://tfhub.dev/tensorflow/lite-model/efficientnet/lite0/uint8/2?lite-format=tflite",
    "https://storage.googleapis.com/tfhub-lite-models/tensorflow/lite-model/efficientnet/lite0/uint8/2/default/1.tflite",
]

EFFICIENTNET_PATH = None

# 1. Check dataset (most reliable on Kaggle)
if DATASET_MODEL_PATH.exists():
    EFFICIENTNET_PATH = DATASET_MODEL_PATH
    print(f"EfficientNet-Lite0 loaded from dataset: {EFFICIENTNET_PATH}")
    print(f"  Size: {EFFICIENTNET_PATH.stat().st_size/1e6:.1f} MB")

# 2. Check working directory
elif EFFICIENTNET_WORKING.exists():
    EFFICIENTNET_PATH = EFFICIENTNET_WORKING
    print(f"EfficientNet-Lite0 found in working dir: {EFFICIENTNET_PATH}")

# 3. Try downloading
else:
    for url in EFFICIENTNET_FALLBACK_URLS:
        try:
            print(f"Trying download: {url[:60]}...")
            urllib.request.urlretrieve(url, EFFICIENTNET_WORKING)
            if EFFICIENTNET_WORKING.stat().st_size > 1_000_000:
                EFFICIENTNET_PATH = EFFICIENTNET_WORKING
                print(f"  Downloaded ({EFFICIENTNET_PATH.stat().st_size/1e6:.1f} MB)")
                break
            else:
                EFFICIENTNET_WORKING.unlink()
        except Exception as e:
            print(f"  Failed: {e}")
            if EFFICIENTNET_WORKING.exists():
                EFFICIENTNET_WORKING.unlink()

if EFFICIENTNET_PATH:
    print(f"\nLiteRT Eye: EfficientNet-Lite0 ready ({EFFICIENTNET_PATH.stat().st_size/1e6:.1f} MB) ✅")
else:
    print("WARNING: EfficientNet-Lite0 not available — visual sensing will use pixel stats fallback")


## Step 3 — LiteRT sensing functions

### Ear: YAMNet audio classification


In [ ]:
import soundfile as sf
from scipy.signal import resample_poly
from math import gcd

# LiteRT interpreter — try ai_edge_litert first, then tflite_runtime
try:
    from ai_edge_litert.interpreter import Interpreter
    print("Using ai_edge_litert")
except ImportError:
    from tflite_runtime.interpreter import Interpreter
    print("Using tflite_runtime")

# YAMNet -> ecological category mapping
# (selected indices from AudioSet 521 classes)
ECO_MAP = {
    "bird_activity":   list(range(0, 23)),     # Bird, Bird vocalization...
    "insect_activity": list(range(71, 76)),    # Insect, Cricket, Bee...
    "rain_signal":     [287, 288, 289],        # Rain, Drizzle...
    "wind_signal":     [361, 362],             # Wind, Rustling leaves
    "frog_signal":     [81, 82, 83],           # Frog, Tree frog...
    "human_noise":     [0, 1, 132, 133, 134,  # Speech, Vehicle...
                        300, 301, 302, 303],
}

YAMNET_SR = 16_000
PATCH_LEN = None  # will be read from model input shape


# Load model once and reuse
_yamnet_interp = None

def _get_yamnet():
    global _yamnet_interp
    if _yamnet_interp is None:
        _yamnet_interp = Interpreter(model_path=str(YAMNET_PATH))
        _yamnet_interp.allocate_tensors()
    return _yamnet_interp

interp  = _get_yamnet()
inp_det = interp.get_input_details()[0]
out_det = interp.get_output_details()[0]
patch_len = inp_det["shape"][0]   # read actual size from model (15600)

def yamnet_scores(wav_path: Path) -> dict:
    """
    Analyse a WAV file with YAMNet.
    Returns averaged class scores across the full audio duration.
    These are PROXY signals — not direct species detections.
    """
    y, sr = sf.read(wav_path, dtype="float32", always_2d=False)
    if y.ndim > 1:
        y = y.mean(axis=1)

    # Resample to 16 kHz
    if sr != YAMNET_SR:
        g = gcd(sr, YAMNET_SR)
        y = resample_poly(y, YAMNET_SR // g, sr // g).astype(np.float32)

    interp  = _get_yamnet()
    inp_det = interp.get_input_details()[0]
    out_det = interp.get_output_details()[0]

    all_scores = []
    for start in range(0, len(y) - patch_len + 1, patch_len):
        patch = y[start : start + patch_len].reshape(inp_det["shape"])
        interp.set_tensor(inp_det["index"], patch)
        interp.invoke()
        all_scores.append(interp.get_tensor(out_det["index"]).flatten())

    if not all_scores:
        return {f"{k}_score": 0.0 for k in ECO_MAP}

    mean_sc = np.mean(all_scores, axis=0)   # 521 class scores

    result = {}
    for cat, indices in ECO_MAP.items():
        valid = [i for i in indices if i < len(mean_sc)]
        result[f"{cat}_score"] = round(float(np.mean(mean_sc[valid])), 4) if valid else 0.0

    result["n_frames"] = len(all_scores)
    return result


# Quick test on one WAV file
if AUDIO_ROOT:
    test_wav = next(AUDIO_ROOT.rglob("*.wav"), None)
    if test_wav:
        print(f"Test: {test_wav.name}")
        t0 = time.time()
        test_scores = yamnet_scores(test_wav)
        print(f"Duration: {time.time()-t0:.1f}s")
        for k, v in test_scores.items():
            print(f"  {k:30s}: {v}")


### Eye: Satellite RGB image analysis (LiteRT / EfficientNet-Lite0)

In [ ]:
def analyze_rgb_litert(tif_path: Path) -> dict:
    """
    Analyse a satellite RGB GeoTIFF using EfficientNet-Lite0 via LiteRT.
    
    Used as a TEXTURE FEATURE EXTRACTOR (not classifier): raw logit activations
    encode visual texture patterns in the satellite image without softmax collapse.
    EfficientNet-Lite0 is not trained on aerial imagery — its class labels are not
    meaningful here, but its convolutional features detect texture differences.
    
    Returns texture energy, spread, and activation ratio as visual proxy signals.
    """
    if not tif_path or not tif_path.exists():
        return {"status": "not_available", "model": "not_available"}
    
    # Pixel stats fallback if LiteRT model unavailable
    if not EFFICIENTNET_PATH or not Path(EFFICIENTNET_PATH).exists():
        try:
            import rasterio
            with rasterio.open(tif_path) as src:
                data = src.read().astype(np.float32)
            r, g, b = data[0], data[1], data[2]
            total = np.nanmean(r) + np.nanmean(g) + np.nanmean(b) + 1e-6
            return {
                "green_dominance_proxy": round(float(np.nanmean(g)) / total, 4),
                "burned_area_signal":    round(float(np.nanmean(r)) / total, 4),
                "model": "pixel_stats_fallback",
            }
        except Exception as e:
            return {"status": f"error:{e}", "model": "pixel_stats_fallback"}

    # ── EfficientNet-Lite0 LiteRT inference ──────────────────────────────────
    try:
        from ai_edge_litert.interpreter import Interpreter
        import rasterio
        from PIL import Image as PILImage

        interp = Interpreter(model_path=str(EFFICIENTNET_PATH))
        interp.allocate_tensors()
        inp_d = interp.get_input_details()[0]
        out_d = interp.get_output_details()[0]

        # Load and preprocess satellite TIF → uint8 224×224
        with rasterio.open(tif_path) as src:
            data = src.read().astype(np.float32)
        
        rgb = np.stack([data[0], data[1], data[2]], axis=-1)
        for c in range(3):
            ch = rgb[:, :, c]
            ch[np.isnan(ch)] = float(np.nanmean(ch))
            lo, hi = float(np.nanmin(ch)), float(np.nanmax(ch))
            rgb[:, :, c] = (ch - lo) / (hi - lo + 1e-6) * 255
        
        pil = PILImage.fromarray(rgb.astype(np.uint8)).resize((224, 224), PILImage.BILINEAR)
        arr = np.array(pil, dtype=np.uint8)[np.newaxis, ...]

        interp.set_tensor(inp_d["index"], arr)
        interp.invoke()
        logits = interp.get_tensor(out_d["index"]).flatten().astype(np.float32)

        # Feature extractor approach: use raw logit distribution as texture proxy
        texture_energy = float(np.sum(logits ** 2))
        texture_spread = float(np.std(logits))
        high_act_ratio = float((logits > logits.mean() + logits.std()).sum()) / len(logits)
        top_logit_idx  = int(logits.argmax())

        return {
            "texture_energy":  round(texture_energy, 2),
            "texture_spread":  round(texture_spread, 4),
            "high_act_ratio":  round(high_act_ratio, 4),
            "top_logit_class": top_logit_idx,
            "model":           "efficientnet_lite0_litert_feature_extractor",
            "note": "Raw logit activations used as texture proxy — not ImageNet class labels",
        }

    except Exception as e:
        return {"status": f"litert_error:{e}", "model": "error"}


## Step 4 — Run LiteRT sensing layer for all sites


In [ ]:
with open(CASES_JSON) as f:
    cases = json.load(f)

print(f"{len(cases)} sites loaded\n")
print("Running LiteRT sensing layer...\n")

sensed_cases = []

for case in cases:
    role    = case["role"]
    site_id = case["site_id"]
    print(f"{'─'*50}")
    print(f"Site: {role} / {site_id}")

    # ── YAMNet: analyse all WAV files ─────────────────────────────────────
    wav_scores_list = []
    for rel_wav in case.get("assets", {}).get("audio_files", []):
        # Try dataset root first, then local path
        wav_candidates = []
        if DATASET_ROOT:
            wav_candidates.append(DATASET_ROOT / rel_wav)
        wav_candidates.append(Path("..") / rel_wav)

        wav_path = next((p for p in wav_candidates if p.exists()), None)
        if wav_path:
            print(f"  YAMNet <- {wav_path.name}", end=" ", flush=True)
            t0 = time.time()
            sc = yamnet_scores(wav_path)
            wav_scores_list.append(sc)
            print(f"({time.time()-t0:.1f}s)")
        else:
            print(f"  WAV not found: {rel_wav}")

    # Average across WAV files
    if wav_scores_list:
        score_keys = [k for k in wav_scores_list[0] if k != "n_frames"]
        yamnet_avg = {
            k: round(float(np.mean([s[k] for s in wav_scores_list])), 4)
            for k in score_keys
        }
        yamnet_avg["files_processed"] = len(wav_scores_list)
    else:
        yamnet_avg = {"status": "no_wav_found"}

    # ── RGB TIF analysis ──────────────────────────────────────────────────
    rgb_rel = case.get("assets", {}).get("rgb_tif")
    if rgb_rel:
        tif_candidates = []
        if DATASET_ROOT:
            tif_candidates.append(DATASET_ROOT / rgb_rel)
        tif_candidates.append(Path("..") / rgb_rel)

        tif_path = next((p for p in tif_candidates if p.exists()), None)
        print(f"  RGB TIF <- {tif_path.name if tif_path else 'not found'}")
        rgb_features = analyze_rgb_litert(tif_path)
    else:
        rgb_features = {"status": "no_rgb_tif"}

    sensed_cases.append({
        **case,
        "litert_sensing": {
            "yamnet_audio":   yamnet_avg,
            "satellite_rgb":  rgb_features,
        },
    })

print(f"\n{len(sensed_cases)} sites sensing complete")


## Step 5 — Build rich multimodal prompt for Gemma 4

Combine YAMNet audio features and RGB pixel features with NDVI + metadata
and pass the full signal set to Gemma 4 for ecological reasoning.


In [ ]:
SYSTEM = (
    "You are an ecological reasoning assistant for Forest Memory — "
    "a multimodal conservation monitoring system for post-fire forest recovery.\n"
    "Input signals come from:\n"
    "  - YAMNet (LiteRT audio model): sound class proxy scores from passive monitoring\n"
    "  - EfficientNet-Lite0 (LiteRT visual model): satellite image texture features\n"
    "  - NDVI: vegetation greenness index\n"
    "  - Site metadata: fire history, land cover, invasive species pressure\n\n"
    "STRICT RULES: Never claim exact species counts. Never predict wildfire. "
    "Always use hedged language: 'proxy signal suggests', 'may indicate', "
    "'consistent with', 'uncertainty remains'. "
    "All scores are relative within a 4-site sample."
)

def build_multimodal_prompt(case: dict, cross_site: dict = None) -> str:
    meta    = case.get("site_metadata", {})
    audio   = case.get("audio", {})
    ndvi    = case.get("ndvi") or {}
    sensing = case.get("litert_sensing", {})
    yamnet  = sensing.get("yamnet_audio", {})
    rgb     = sensing.get("satellite_rgb", {})
    flags   = [k for k, v in case.get("interpretation_flags", {}).items() if v]

    def _sc(col):
        d = audio.get(col, {})
        return f"{d['mean']:.1f}/100" if isinstance(d, dict) and d.get("mean") else "N/A"

    def _rgb_block(rgb_dict):
        model = rgb_dict.get("model", "not_available")
        if model == "not_available":
            return "  visual sensing : not available (no satellite TIF)"
        if model == "pixel_stats_fallback":
            return (
                f"  green_dominance_proxy : {rgb_dict.get('green_dominance_proxy', 'N/A')}\n"
                f"  burned_area_signal    : {rgb_dict.get('burned_area_signal', 'N/A')}\n"
                f"  model                 : pixel_stats_fallback"
            )
        # EfficientNet texture features
        return (
            f"  texture_energy   : {rgb_dict.get('texture_energy', 'N/A')}  (higher = more complex texture)\n"
            f"  texture_spread   : {rgb_dict.get('texture_spread', 'N/A')}  (std of logit activations)\n"
            f"  high_act_ratio   : {rgb_dict.get('high_act_ratio', 'N/A')}  (fraction of strongly activated features)\n"
            f"  model            : EfficientNet-Lite0 LiteRT (texture feature extractor, not ImageNet classifier)"
        )

    # Build relative cross-site context block
    cross_block = ""
    if cross_site:
        ndvi_val = ndvi.get("mean_ndvi")
        vit_val  = (audio.get("bioacoustic_vitality_score") or {}).get("mean")

        def _rank_label(val, sorted_vals, higher_is_better=True):
            if val is None or not sorted_vals:
                return "N/A"
            rank = sorted(sorted_vals).index(val) + 1
            n = len(sorted_vals)
            if higher_is_better:
                if rank == n:   return "HIGHEST among all 4 sites"
                elif rank == 1: return "LOWEST among all 4 sites"
                else:           return f"rank {rank}/{n} among sites"
            else:
                if rank == 1:   return "HIGHEST among all 4 sites"
                elif rank == n: return "LOWEST among all 4 sites"
                else:           return f"rank {rank}/{n} among sites"

        ndvi_label = _rank_label(ndvi_val, cross_site.get("ndvi_values", []))
        vit_label  = _rank_label(vit_val,  cross_site.get("vit_values",  []))

        ndvi_range = f"{cross_site['ndvi_min']:.3f} – {cross_site['ndvi_max']:.3f}" if cross_site.get("ndvi_min") is not None else "N/A"
        vit_range  = f"{cross_site['vit_min']:.1f} – {cross_site['vit_max']:.1f}" if cross_site.get("vit_min") is not None else "N/A"

        cross_block = f"""
── CROSS-SITE CONTEXT (4-site sample) ─────────────────
NDVI range across sites : {ndvi_range}  →  this site is {ndvi_label}
Vitality range          : {vit_range} /100  →  this site is {vit_label}
"""

    return f"""{SYSTEM}

=== SITE: {case['role'].upper()} | {case['site_id']} ==={cross_block}
── ECOSYSTEM CONTEXT ─────────────────────────────────
Land cover  : {meta.get('land_cover_class', '?')}
Fire history: {meta.get('fire_class', '?')}
Veg age     : {meta.get('field_veld_age', '?')}
Aliens      : {meta.get('field_aliens_within_20m', '?')}
Season      : {meta.get('campaign', '?')}

── LiteRT / YAMNet AUDIO SENSING (proxy scores, 0–1) ──
bird_activity_score   : {yamnet.get('bird_activity_score', 'N/A')}
insect_activity_score : {yamnet.get('insect_activity_score', 'N/A')}
rain_signal_score     : {yamnet.get('rain_signal_score', 'N/A')}
wind_signal_score     : {yamnet.get('wind_signal_score', 'N/A')}
human_noise_score     : {yamnet.get('human_noise_score', 'N/A')}
frog_signal_score     : {yamnet.get('frog_signal_score', 'N/A')}
files_processed       : {yamnet.get('files_processed', '?')}

── LiteRT / EfficientNet-Lite0 VISUAL SENSING ──────────
{_rgb_block(rgb)}

── NDVI (Sentinel-2, 500m buffer, Oct–Dec 2023) ────────
mean_ndvi  : {ndvi.get('mean_ndvi', 'N/A')}
std_ndvi   : {ndvi.get('std_ndvi', 'N/A')}

── ACOUSTIC VITALITY PROXIES (FFT-based, 0–100 scale) ──
bioacoustic_vitality  : {_sc('bioacoustic_vitality_score')}
acoustic_richness     : {_sc('acoustic_richness_score')}
human_disturbance     : {_sc('human_disturbance_proxy')}

── INTERPRETATION FLAGS ────────────────────────────────
{', '.join(flags) if flags else 'none'}

Write an ecological resilience report with EXACTLY these 4 sections (2-3 sentences each).
Use careful proxy language throughout.

## Vegetation Signal
## Acoustic Signal
## Multimodal Tension
## Recovery Outlook
"""

# Preview prompt for the first site
print(build_multimodal_prompt(sensed_cases[0])[:1500])
print("...")


## Step 6 — Ecological reasoning with Gemma 4

**Two modes supported:**
- **LiteRT mode** (if the Gemma 4 `.task` file is added to Kaggle) → fully offline
- **API mode** (fallback) → requires a Google AI Studio API key


In [ ]:

gemma_llm  = None
gemma_mode = "none"

try:
    import google.genai as genai
    from kaggle_secrets import UserSecretsClient
    api_key = UserSecretsClient().get_secret("GOOGLE_API_KEY")
    client = genai.Client(api_key=api_key)
    gemma_model_name = "gemma-4-31b-it"
    gemma_mode = "api"
    print(f"Connected: {gemma_model_name}")
except Exception as e:
    print(f"Error: {e}")

print(f"Active mode: {gemma_mode}")


In [ ]:
gemma_model_name = "gemma-4-26b-a4b-it"  # lighter MoE variant

# Pre-compute cross-site NDVI and vitality ranges for relative context
ndvi_vals = [c["ndvi"]["mean_ndvi"] for c in sensed_cases if c.get("ndvi")]
vit_vals  = [(c["audio"].get("bioacoustic_vitality_score") or {}).get("mean")
             for c in sensed_cases]
vit_vals  = [v for v in vit_vals if v is not None]

cross_site_ctx = {
    "ndvi_values": ndvi_vals,
    "ndvi_min":    min(ndvi_vals) if ndvi_vals else None,
    "ndvi_max":    max(ndvi_vals) if ndvi_vals else None,
    "vit_values":  vit_vals,
    "vit_min":     min(vit_vals)  if vit_vals  else None,
    "vit_max":     max(vit_vals)  if vit_vals  else None,
}
print("Cross-site context:")
print(f"  NDVI range    : {cross_site_ctx['ndvi_min']:.3f} – {cross_site_ctx['ndvi_max']:.3f}")
print(f"  Vitality range: {cross_site_ctx['vit_min']:.1f} – {cross_site_ctx['vit_max']:.1f}")
print()


def generate_report(case: dict) -> dict:
    prompt = build_multimodal_prompt(case, cross_site=cross_site_ctx)
    t0 = time.time()

    if gemma_mode == "none":
        return {"role": case["role"], "site_id": case["site_id"],
                "report": "[Gemma not loaded]", "mode": "none", "latency_s": 0}

    for attempt in range(3):
        try:
            response = client.models.generate_content(
                model=gemma_model_name,
                contents=prompt,
                config=genai.types.GenerateContentConfig(
                    temperature=0.3, max_output_tokens=600
                ),
            )
            return {
                "role":      case["role"],
                "site_id":   case["site_id"],
                "report":    response.text,
                "mode":      gemma_mode,
                "latency_s": round(time.time() - t0, 2),
            }
        except Exception as e:
            print(f"  attempt {attempt+1} failed: {e}")
            time.sleep(5)

    return {"role": case["role"], "site_id": case["site_id"],
            "report": f"[Error after 3 attempts]", "mode": gemma_mode,
            "latency_s": round(time.time() - t0, 2)}


print("Generating reports...\n")
reports = []
EMOJI = {"healthy_baseline": "🌿", "burned_recovering": "🔥",
         "invasive_disturbed": "⚠️", "wet_dry_pair_complement": "💧"}

for case in sensed_cases:
    e = EMOJI.get(case["role"], "📍")
    print(f"{e} {case['role']} ...", end=" ", flush=True)
    r = generate_report(case)
    reports.append(r)
    print(f"{r['latency_s']}s [{r['mode']}]")
    time.sleep(3)

print("\nAll reports complete")


## Step 7 — Display reports


In [ ]:
for r in reports:
    e = EMOJI.get(r["role"], "📍")
    badge = "🟢 OFFLINE" if r["mode"] == "litert-offline" else "🔵 API"
    display(Markdown(
        f"---\n## {e} {r['role'].replace('_',' ').title()}  {badge}\n"
        f"`{r['site_id']}` | latency: **{r['latency_s']}s** | mode: `{r['mode']}`\n\n"
        + r["report"]
        + "\n\n> *All values are proxy signals — not direct species detections or wildfire predictions.*"
    ))


## Step 8 — Save outputs and summary


In [ ]:
out_file = OUT_DIR / "litert_multimodal_reports.json"
with open(out_file, "w") as f:
    json.dump({
        "pipeline_mode": gemma_mode,
        "litert_audio_model": "YAMNet (AudioSet, 521 classes)",
        "litert_visual_model": "EfficientNet-Lite0 (texture feature extractor)",
        "gemma_model": gemma_model_name,
        "reports": reports,
    }, f, indent=2)
print(f"Saved -> {out_file}")

print("\n" + "="*65)
print("SUMMARY — LiteRT Sensing + Gemma 4 Reasoning")
print("="*65)
print(f"{'Role':35s} {'Latency':8s} {'Mode'}")
print("-"*65)
for r in reports:
    print(f"{r['role']:35s} {r['latency_s']:5.1f}s   {r['mode']}")

wav_count  = sum(c.get("litert_sensing", {}).get("yamnet_audio", {}).get("files_processed", 0)
                 for c in sensed_cases)
rgb_ok     = sum(1 for c in sensed_cases
                 if c.get("litert_sensing", {}).get("satellite_rgb", {}).get("model")
                 == "efficientnet_lite0_litert_feature_extractor")
rgb_fallbk = sum(1 for c in sensed_cases
                 if c.get("litert_sensing", {}).get("satellite_rgb", {}).get("model")
                 == "pixel_stats_fallback")

print(f"""
Architecture summary:
  YAMNet (LiteRT ear)        -> {wav_count} WAV files analysed, {len(reports)} site averages
  EfficientNet-Lite0 (LiteRT eye) -> {rgb_ok} TIFs processed via LiteRT, {rgb_fallbk} via pixel fallback
  Gemma 4 ({gemma_mode})     -> {len(reports)} ecological reports generated
""")


---

## Kaggle setup instructions

```
1. Open a new Kaggle Notebook
2. + Add Data -> "New Dataset" -> upload your project folder:
      data/bioscape/audio/
      data/bioscape/sattelite/
      outputs/forest_memory_cases.json
      src/
3. + Add Model -> "google/gemma4/litert" -> select the smallest variant
4. Session Options -> Internet: OFF (for offline demo)
5. Run All -> YAMNet downloads automatically, pipeline runs end-to-end
```

**Requirements for the LiteRT prize ($10,000):**
- Real audio classification via YAMNet LiteRT
- Offline inference via Gemma 4 LiteRT
- Runs with internet turned off (demonstrate in video)
- Edge hardware deployment blueprint
